# 05 — SDMX Export (Phase 5)

Converts the merged global outputs into SDMX-CSV long format for submission to UN statistical data portals. Produces three streams: hazard, exposure, population.

## Inputs

- Global CSVs produced by notebook 02

## Outputs

`{output_folder}/sdmx/{hazard|exposure|population}/CSV/HAZARD_SDMX_*.csv`
`{output_folder}/sdmx/{hazard|exposure|population}/CSV/HAZARD_SDMX_*.xlsx`

## Execution order

Run **after** notebook 02.


In [ ]:
# ============================================================
# CCRI Hazard Statistics Processing per country administrative unit 2
# SDMX export for hazard, exposure, and population
# Author: Angelly Pugliese, Ph.D.
# Date: April 2026
# ============================================================

In [ ]:
# ============================================================
# Imports
# ============================================================
import sys
from pathlib import Path

# Anchor everything to the project root so config/bounds/lib resolve regardless of CWD.
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pipeline" / "settings.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from pipeline.settings import Settings
OUTPUT_ROOT = Settings.from_env().storage_root   # notebooks/output unless GCHD_STORAGE_ROOT is set
LIB_DIR = PROJECT_ROOT / "lib"
CONFIG_DIR = PROJECT_ROOT / "config"
BOUNDS_DIR = PROJECT_ROOT / "bounds"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

import pandas as pd
import geopandas as geopd
import ee
import os
import shapely
from GEE_functions import GEEUtils 
from GEE_functions import Hazard
from indicator_functions import IndicatorUtils
import datetime as dt
import csv

import numpy as np
pd.set_option('display.max_columns', 500)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# ============================================================
# Authenticate and initialize the Earth Engine library
# Define GEE asset path for UNICEF CCRI data
# ============================================================
ee.Authenticate()
ee.Initialize(project="unicef-ccri")
unicef_data_source_path = "projects/unicef-ccri/assets"
utils = GEEUtils(ee)
indicator_utils = IndicatorUtils()

In [ ]:
prod_date = dt.date.today().isoformat()

In [ ]:
hazard_info = pd.read_json(CONFIG_DIR / "hazard_info.json")
hazard_list = [
    Hazard.from_dict(row, asset_prefix=unicef_data_source_path)
    for row in hazard_info.to_dict(orient="records")
]

# Now hazard_list contains Hazard objects
for index, hazard in enumerate(hazard_list):
    print(f"{index}. {hazard.name}, {hazard.code}, {hazard.asset}, {hazard.threshold}, {hazard.threshold_operation}")

In [ ]:
output_folder = OUTPUT_ROOT
data_base_path = f"{output_folder}/global/CSV"
prod_date = "2026-08-31"
# TIME_PERIOD is read per-row from the source CSV's own `time_period` column so the
# SDMX reference period always matches the data (the CSVs carry 2025), instead of a
# hardcoded value that silently drifts from the source.
TIME_PERIOD_FALLBACK = "2025"

# ------------------------------------------------------------------
# Population source.
# Population must NOT be read from a hazard CSV: those blank pop_* for any unit
# the hazard flags, so the stream would inherit that hazard's flags. The loop
# builds population from the FIRST hazard processed (river_flood_100yr), which
# flags Fiji as Insufficient Data -- silently dropping Fiji's entire population.
# The boundary asset carries the 6 population classes for every admin2 unit
# independently of any hazard, so it is the correct source.
POPULATION_TABLE = f"{output_folder}/aux/admin2_population.csv"

# Countries whose population is deliberately published BLANK. These are flags on
# the unit itself (not on a hazard), mirroring notebook 02:
#   non_official_codes + sovereignty_unsettled_countries -> "Not Applicable"
#   id_countries_insufficient_data                       -> "Insufficient Data"
# Countries flagged per-hazard (e.g. FJI for riverine flood) keep their population.
POP_BLANK_COUNTRIES = (
    ['ALA_V1', 'CCK_V1', 'CXR_V1', 'ESH_V1', 'GGY_V1', 'JEY_V1', 'NFK_V1', 'PCN_V1', 'SJM_V1']
    + ['EGY1_V1', 'SDN1_V1', 'SSD1_V1', 'xAB_V1', 'xAC_V1', 'xAP_V1', 'xJK_V1', 'xJL_V1',
       'xPI_V1', 'xRI_V1', 'xSI_V1', 'xSK_V1', 'xSR_V1', 'xxx_V1']
    + ['NIC_V1', 'PSE_V1', 'CUB_V2']
)

def build_population_table(path):
    """Pull the 6 population classes per admin2 unit from the boundary asset.

    Fetched per country because a single getInfo() over the whole collection
    exceeds GEE's 5000-element limit; geometry is dropped so it stays light.
    Large countries are split further by adm1 if a country still exceeds it.
    """
    fc = ee.FeatureCollection(f"{unicef_data_source_path}/{utils.ADMIN2_ASSET}")
    cols = ["adm2_ucode", "pop_total", "pop_total_male", "pop_total_female",
            "pop_under_18_total", "pop_under_18_male", "pop_under_18_female"]

    ucodes = sorted(utils.get_countries_to_process(None))
    print(f"  fetching population for {len(ucodes)} countries from "
          f"{utils.ADMIN2_ASSET} ...")

    rows = []
    for n, uc in enumerate(ucodes, 1):
        sub = fc.filter(ee.Filter.eq("adm0_ucode", uc)).select(cols, retainGeometry=False)
        try:
            feats = sub.getInfo()["features"]
        except Exception:
            # country over the element cap -> page it by adm1
            feats = []
            a1s = (fc.filter(ee.Filter.eq("adm0_ucode", uc))
                     .aggregate_array("adm1_ucode").distinct().getInfo())
            for a1 in a1s:
                s2 = fc.filter(ee.Filter.eq("adm1_ucode", a1)).select(cols, retainGeometry=False)
                feats += s2.getInfo()["features"]
        rows += [f["properties"] for f in feats]
        if n % 50 == 0:
            print(f"    {n}/{len(ucodes)} countries, {len(rows):,} units")

    df = pd.DataFrame(rows).drop_duplicates("adm2_ucode")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)
    print(f"  wrote {len(df):,} admin2 units -> {path}")
    return df


# adm2_ucode -> {sdmx population column: value}, from the boundary asset.
# Built once and cached on disk; rebuilt automatically if the file is missing.
if not os.path.exists(POPULATION_TABLE):
    print(f"population table not found at {POPULATION_TABLE}; building from GEE")
    _pop_raw = build_population_table(POPULATION_TABLE)
else:
    _pop_raw = pd.read_csv(POPULATION_TABLE)
_POP_RENAME = {
    "pop_total": "pop_total",
    "pop_total_male": "pop_total_m",
    "pop_total_female": "pop_total_f",
    "pop_under_18_total": "u18_pop_total",
    "pop_under_18_male": "u18_pop_m",
    "pop_under_18_female": "u18_pop_f",
}
_pop_raw = _pop_raw.rename(columns=_POP_RENAME)
POPULATION_BY_UCODE = _pop_raw.set_index("adm2_ucode")[list(_POP_RENAME.values())].to_dict("index")
print(f"population table: {len(POPULATION_BY_UCODE):,} admin2 units "
      f"({len(POP_BLANK_COUNTRIES)} countries published blank)")
is_draft = True

# Create the folder to store the SDMX
sdmx_hazard_folder_path = f"{output_folder}/sdmx/hazard"
sdmx_exposure_folder_path = f"{output_folder}/sdmx/exposure"
sdmx_population_folder_path = f"{output_folder}/sdmx/population"
os.makedirs(f"{sdmx_hazard_folder_path}/CSV", exist_ok=True)
os.makedirs(f"{sdmx_exposure_folder_path}/CSV", exist_ok=True)
os.makedirs(f"{sdmx_population_folder_path}/CSV", exist_ok=True)

# create Hazards SDMX structure
# UNIT_MEASURE for the hazs_* statistics is the hazard's OWN physical unit (m, m/s,
# PGA (g), ug/m3 ...), taken from `unit` in config/hazard_info.json -- a single
# "NUMBER" code would make depth-in-metres and ug/m3 indistinguishable to a
# consumer. HAZARD_UNIT is a sentinel resolved per hazard inside the loop below.
# The g/c/r indices stay INDEX: a normalised 0-10 score is unitless by construction.
HAZARD_UNIT = "__HAZARD_UNIT__"

hazards_variable = {
    "hazs_mean": { "unit": HAZARD_UNIT, "measure": "HAZARD_MEAN", },
    "hazs_std": { "unit": HAZARD_UNIT, "measure": "HAZARD_STD" },
    "hazs_max": { "unit": HAZARD_UNIT, "measure": "HAZARD_MAX" },
    "hazs_min": { "unit": HAZARD_UNIT, "measure": "HAZARD_MIN" },
    "hazs_median": { "unit": HAZARD_UNIT, "measure": "HAZARD_MEDIAN" },
    # Hazard index at all three normalisation scopes (was: country only).
    "gindex": { "unit": "INDEX", "measure": "HAZARD_GLOBAL_INDEX" },
    "cindex": { "unit": "INDEX", "measure": "HAZARD_COUNTRY_INDEX" },
    "rindex": { "unit": "INDEX", "measure": "HAZARD_REGIONAL_INDEX" },
}

# ------------------------------------------------------------------
# Exposure index / class grid.
# The source CSVs carry these for every combination of
#     scope  (g=global, c=country, r=regional)
#   x sex    (''=total, m=male, f=female)
#   x age    (''=total population, 'u18_'=children 0-17)
# = 3 x 3 x 2 = 18 index + 18 class columns.
# They belong in the EXPOSURE stream (not hazard) because only that stream carries
# the SEX and AGE dimensions needed to tell them apart. Previously only cexpi/cexpc
# (country scope, total population) were exported and the other 34 were dropped.
# ------------------------------------------------------------------
SCOPE_MEASURE = {
    "g": ("GLOBAL_EXPOSURE_INDEX",   "GLOBAL_EXPOSURE_CLASS"),
    "c": ("COUNTRY_EXPOSURE_INDEX",  "COUNTRY_EXPOSURE_CLASS"),
    "r": ("REGIONAL_EXPOSURE_INDEX", "REGIONAL_EXPOSURE_CLASS"),
}

exposure_index_variables = {}
for _age_prefix, _age in (("", "_T"), ("u18_", "Y0T17")):
    for _scope, (_mi, _mc) in SCOPE_MEASURE.items():
        for _sex_infix, _sex in (("", "_T"), ("m", "M"), ("f", "F")):
            exposure_index_variables[f"{_age_prefix}{_scope}{_sex_infix}expi"] = {
                "unit": "INDEX", "measure": _mi, "sex": _sex, "age": _age,
            }
            exposure_index_variables[f"{_age_prefix}{_scope}{_sex_infix}expc"] = {
                "unit": "CLASS", "measure": _mc, "sex": _sex, "age": _age,
            }
sdmx_hazards_structure = {}
dims_hazards = [
    "DATAFLOW", "REF_AREA", "INDICATOR", "MEASURE", "ADMIN_LEVEL", "ISO3_PARENT", 
    "TIME_PERIOD", "OBS_VALUE", "UNIT_MEASURE", "REGION", "STATUS"
] 

# create Exposure SDMX structure
# SEX/AGE are declared per column rather than inferred from substrings ("f" in col
# is true for any column containing an f), removing that fragility.
# UNIT_MEASURE reflects what the number actually is:
#   PEOPLE     - a headcount of exposed persons (absolute exposure)
#   PERCENTAGE - share of the population exposed (relative exposure, already 0-100)
#   INDEX      - normalised 0-10 exposure index
#   CLASS      - discrete 0-5 exposure class
# SEX/AGE are declared per column rather than inferred from substrings ("f" in col
# is true for any column containing an f), removing that fragility.
exposure_variables = {
    "aexp":      { "unit": "PEOPLE", "measure": "ABSOLUTE_EXPOSURE", "sex": "_T", "age": "_T" },
    "amexp":     { "unit": "PEOPLE", "measure": "ABSOLUTE_EXPOSURE", "sex": "M",  "age": "_T" },
    "afexp":     { "unit": "PEOPLE", "measure": "ABSOLUTE_EXPOSURE", "sex": "F",  "age": "_T" },
    "u18_aexp":  { "unit": "PEOPLE", "measure": "ABSOLUTE_EXPOSURE", "sex": "_T", "age": "Y0T17" },
    "u18_amexp": { "unit": "PEOPLE", "measure": "ABSOLUTE_EXPOSURE", "sex": "M",  "age": "Y0T17" },
    "u18_afexp": { "unit": "PEOPLE", "measure": "ABSOLUTE_EXPOSURE", "sex": "F",  "age": "Y0T17" },

    "rexp":      { "unit": "PERCENTAGE", "measure": "RELATIVE_EXPOSURE", "sex": "_T", "age": "_T" },
    "rmexp":     { "unit": "PERCENTAGE", "measure": "RELATIVE_EXPOSURE", "sex": "M",  "age": "_T" },
    "rfexp":     { "unit": "PERCENTAGE", "measure": "RELATIVE_EXPOSURE", "sex": "F",  "age": "_T" },
    "u18_rexp":  { "unit": "PERCENTAGE", "measure": "RELATIVE_EXPOSURE", "sex": "_T", "age": "Y0T17" },
    "u18_rmexp": { "unit": "PERCENTAGE", "measure": "RELATIVE_EXPOSURE", "sex": "M",  "age": "Y0T17" },
    "u18_rfexp": { "unit": "PERCENTAGE", "measure": "RELATIVE_EXPOSURE", "sex": "F",  "age": "Y0T17" },
}
# the 36-column index/class grid is emitted into the same exposure stream
exposure_variables.update(exposure_index_variables)
# the 36-column index/class grid is emitted into the same exposure stream
exposure_variables.update(exposure_index_variables)
dims_exposure = [
    "DATAFLOW", "REF_AREA", "INDICATOR", "MEASURE", "SEX", "AGE", "ADMIN_LEVEL", "ISO3_PARENT", 
    "TIME_PERIOD", "OBS_VALUE", "UNIT_MEASURE", "REGION", "STATUS"
]

# create Population SDMX structure
population_variables = {
    "pop_total":     { "unit": "PEOPLE", "measure": "POP_TOTAL",     "sex": "_T", "age": "_T" },
    "pop_total_m":   { "unit": "PEOPLE", "measure": "POP_M",         "sex": "M",  "age": "_T" },
    "pop_total_f":   { "unit": "PEOPLE", "measure": "POP_F",         "sex": "F",  "age": "_T" },
    "u18_pop_total": { "unit": "PEOPLE", "measure": "U18_POP_TOTAL", "sex": "_T", "age": "Y0T17" },
    "u18_pop_m":     { "unit": "PEOPLE", "measure": "U18_POP_M",     "sex": "M",  "age": "Y0T17" },
    "u18_pop_f":     { "unit": "PEOPLE", "measure": "U18_POP_F",     "sex": "F",  "age": "Y0T17" }
}
# NOTE: no STATUS on population. The status flag describes availability of the
# HAZARD observation (No Data / Insufficient Data / Not Applicable); population is
# a denominator that exists independently of any hazard and is populated even on
# rows where hazs_* and aexp are null. Carrying the hazard status onto population
# would wrongly mark valid population figures as unavailable.
dims_population = [
    "DATAFLOW", "REF_AREA", "INDICATOR", "SEX", "AGE", "ADMIN_LEVEL", "ISO3_PARENT",
    "TIME_PERIOD", "OBS_VALUE", "UNIT_MEASURE", "REGION"
]


population_calculated = False
for index, hazard in enumerate(hazard_list):
    print(f"Processing SDMX files for hazard {index}: {hazard.name} ({hazard.code})")    
    # Create new SDMX structures for the hazard
    sdmx_hazards_structure = {dim: [] for dim in dims_hazards}  
    sdmx_exposure_structure = {dim: [] for dim in dims_exposure}    
    sdmx_population_structure = {dim: [] for dim in dims_population}    

    excel_path = f"{data_base_path}/{hazard.code}_{prod_date}.csv"
    hazard_df = pd.read_csv(excel_path, index_col=None)
    hazard_df = hazard_df.replace(np.nan, None)
    hazard_name = hazard.name
    hazard_code = hazard.code
    hazard_indicator = hazard.indicator
    # Physical unit for this hazard's hazs_* rows. Fall back to NUMBER only if the
    # hazard has no `unit` in hazard_info.json, so a missing entry is visible as the
    # old generic code rather than an empty UNIT_MEASURE.
    hazard_unit = hazard.unit or "NUMBER"
    admin_level = "2"
    draft_text = "_DRAFT" if is_draft else ""

    for df_row in hazard_df.itertuples(index=False):
        adm2_ucode = getattr(df_row, "adm2_ucode")
        iso3 = getattr(df_row, "iso3")
        region = getattr(df_row, "unicef_region")
        # Blank/NaN status means the observation is usable. Normalise NaN -> "" so the
        # SDMX column is empty rather than the literal string "nan".
        status = getattr(df_row, "status")
        if status is None or (isinstance(status, float) and pd.isna(status)):
            status = ""
        # Reference period comes from the row itself; fall back only if absent.
        time_period = getattr(df_row, "time_period", None) or TIME_PERIOD_FALLBACK

        # Populate the SDMX structure for hazards
        for col, indicator in hazards_variable.items():
            sdmx_hazards_structure["DATAFLOW"].append(f"UNICEF{draft_text}:HAZARD_INTENSITY(1.0)")
            sdmx_hazards_structure["REF_AREA"].append(adm2_ucode)
            sdmx_hazards_structure["INDICATOR"].append(f"HAZARD_{hazard_indicator}")
            sdmx_hazards_structure["MEASURE"].append(indicator["measure"])
            sdmx_hazards_structure["ADMIN_LEVEL"].append(admin_level)
            sdmx_hazards_structure["ISO3_PARENT"].append(iso3)
            sdmx_hazards_structure["TIME_PERIOD"].append(time_period)
            sdmx_hazards_structure["OBS_VALUE"].append(getattr(df_row, col))
            sdmx_hazards_structure["UNIT_MEASURE"].append(
                hazard_unit if indicator["unit"] == HAZARD_UNIT else indicator["unit"])
            sdmx_hazards_structure["REGION"].append(region)
            sdmx_hazards_structure["STATUS"].append(status)

        # Populate the SDMX structure for exposure
        for col, indicator in exposure_variables.items():
            sdmx_exposure_structure["DATAFLOW"].append(f"UNICEF{draft_text}:HAZARD_EXPOSURE(1.0)")
            sdmx_exposure_structure["REF_AREA"].append(adm2_ucode)
            sdmx_exposure_structure["INDICATOR"].append(f"HAZARD_EXPOSURE_{hazard_indicator}")
            sdmx_exposure_structure["MEASURE"].append(indicator["measure"])

            sdmx_exposure_structure["SEX"].append(indicator["sex"])
            sdmx_exposure_structure["AGE"].append(indicator["age"])
            
            sdmx_exposure_structure["ADMIN_LEVEL"].append(admin_level)
            sdmx_exposure_structure["ISO3_PARENT"].append(iso3)
            sdmx_exposure_structure["TIME_PERIOD"].append(time_period)
            sdmx_exposure_structure["OBS_VALUE"].append(getattr(df_row, col))
            sdmx_exposure_structure["UNIT_MEASURE"].append(indicator["unit"])
            sdmx_exposure_structure["REGION"].append(region)
            sdmx_exposure_structure["STATUS"].append(status)

        # Populate the SDMX structure for population.
        # Values come from the boundary asset (POPULATION_BY_UCODE), not from this
        # hazard's row, so the population stream carries no hazard-specific gaps.
        if not population_calculated:
            _adm0 = getattr(df_row, "adm0_ucode", None)
            _pop = POPULATION_BY_UCODE.get(adm2_ucode, {})
            _blank_pop = _adm0 in POP_BLANK_COUNTRIES
            for col, indicator in population_variables.items():
                sdmx_population_structure["DATAFLOW"].append(f"UNICEF{draft_text}:HAZARD_POPULATION(1.0)")
                sdmx_population_structure["REF_AREA"].append(adm2_ucode) 
                sdmx_population_structure["INDICATOR"].append("HAZARD_POPULATION")
                
                sdmx_population_structure["SEX"].append(indicator["sex"])
                sdmx_population_structure["AGE"].append(indicator["age"])

                sdmx_population_structure["ADMIN_LEVEL"].append(admin_level)
                sdmx_population_structure["ISO3_PARENT"].append(iso3)
                sdmx_population_structure["TIME_PERIOD"].append(time_period)
                sdmx_population_structure["OBS_VALUE"].append(
                    None if _blank_pop else _pop.get(col))
                sdmx_population_structure["UNIT_MEASURE"].append(indicator["unit"])
                sdmx_population_structure["REGION"].append(region)

    # Save hazard SDMX for all countries for specific hazard.
    # CSV only: the exposure stream is ~2M rows per hazard (41k admin2 x 48
    # measures), which exceeds Excel's 1,048,576-row sheet limit and makes
    # .to_excel() the dominant cost of the run.
    sdmx_hazard_df = pd.DataFrame(sdmx_hazards_structure)
    path_csv = f"{sdmx_hazard_folder_path}/CSV/HAZARD_SDMX_hazard_{hazard_code}_{prod_date}.csv"
    sdmx_hazard_df.to_csv(path_csv, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

    # Save exposure SDMX for all countries for specific hazard
    sdmx_exposure_df = pd.DataFrame(sdmx_exposure_structure)
    path_csv = f"{sdmx_exposure_folder_path}/CSV/HAZARD_SDMX_exposure_{hazard_code}_{prod_date}.csv"
    sdmx_exposure_df.to_csv(path_csv, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)
    
    # The population SDMX is only done once, as the population data is the same for all hazards. 
    # It is calculated during the first iteration and then skipped for the rest of the hazards.
    if not population_calculated:
        population_calculated = True
        sdmx_population_df = pd.DataFrame(sdmx_population_structure)

        path_csv = f"{sdmx_population_folder_path}/CSV/HAZARD_SDMX_population_{prod_date}.csv"
        sdmx_population_df.to_csv(path_csv, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)
